In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys
from ultralytics import YOLO
import ultralytics.nn.modules as modules
import ultralytics.nn.modules.block as block
import ultralytics.nn.tasks as tasks


# ============================ 
# SWIN TRANSFORMER
# ============================ 

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        attn = self.softmax(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x

class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4.0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim)
        )

    def forward(self, x):
        H, W = self.H, self.W
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = F.pad(x, (0, 0, pad_l, pad_r, pad_t, pad_b))
        _, Hp, Wp, _ = x.shape
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x
        x_windows = self.window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = self.window_reverse(attn_windows, self.window_size, Hp, Wp)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W,:].contiguous()
        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

    def window_partition(self, x, window_size):
        B, H, W, C = x.shape
        x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
        windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
        return windows

    def window_reverse(self, windows, window_size, H, W):
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
        return x


class _SwinTransformerCore(nn.Module):
    def __init__(self, c1, c2, num_heads=4, depth=2, window_size=7):
        super().__init__()
        self.c2 = c2
        self.input_proj = nn.Conv2d(c1, c2, kernel_size=1)
        self.blocks = nn.ModuleList()
        for i in range(depth):
            shift_size = 0 if (i % 2 == 0) else window_size // 2
            self.blocks.append(
                SwinTransformerBlock(dim=c2, num_heads=num_heads, 
                                   window_size=window_size, shift_size=shift_size)
            )
        self.norm = nn.LayerNorm(c2)
        self.output_proj = nn.Conv2d(c2, c2, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.input_proj(x)
        x = x.flatten(2).transpose(1, 2)
        for block in self.blocks:
            block.H, block.W = H, W
            x = block(x)
        x = self.norm(x)
        x = x.transpose(1, 2).reshape(B, self.c2, H, W)
        x = self.output_proj(x)
        return x


# ============================ 
# UAVSwin (Đổi tên từ SwinTransformer)
# ============================ 
# Giữ nguyên các class WindowAttention, SwinTransformerBlock, _SwinTransformerCore...

class UAVSwin(nn.Module):
    # Thêm in_channels, out_channels rõ ràng vào signature để tránh nhầm lẫn
    def __init__(self, c1, c2, *args):
        super().__init__()
        # args từ YAML sẽ là [num_heads, depth, window_size]
        # Nhưng đôi khi Ultralytics sẽ truyền: (c1, c2, num_heads, ...)
        # Hoặc: (c1, num_heads, ...) -> lúc này c2 sẽ là num_heads, args sẽ bị lệch.
        
        # --- DEBUG ĐỂ TÌM NGUYÊN NHÂN ---
        print(f"DEBUG INIT: c1={c1}, c2={c2}, args={args}")
        
        # Xử lý linh hoạt args
        if len(args) == 3: # Trường hợp chuẩn: c1, c2, n_heads, depth, win_size
             self.num_heads, self.depth, self.window_size = args
        elif len(args) == 2: # Trường hợp bị lệch: c2 lấy mất n_heads
             self.num_heads = c2
             self.depth, self.window_size = args
             c2 = c1 # Output channels = Input channels (Backbone thường giữ nguyên hoặc theo YAML)
        else:
             # Fallback mặc định
             self.num_heads = args[0] if len(args) > 0 else 4
             self.depth = args[1] if len(args) > 1 else 2
             self.window_size = args[2] if len(args) > 2 else 7

        self.c2 = c1 # Swin thường giữ nguyên channels, hoặc bạn có thể dùng c2 nếu muốn đổi
        
        print(f"🔧 UAVSwin setup: c1={c1} -> c2={self.c2} | Heads={self.num_heads}, Depth={self.depth}")
        
        self.core = _SwinTransformerCore(
            c1=c1,
            c2=self.c2, # Lưu ý dùng self.c2
            num_heads=self.num_heads,
            depth=self.depth,
            window_size=self.window_size
        )

    def forward(self, x):
        return self.core(x)


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys
from ultralytics import YOLO
import ultralytics.nn.modules as modules
import ultralytics.nn.modules.block as block
import ultralytics.nn.tasks as tasks

# ✅ XÓA CACHE CŨ ĐỂ TRÁNH XUNG ĐỘT
for module_list in [modules, block, tasks]:
    if hasattr(module_list, 'UAVSwin'):
        delattr(module_list, 'UAVSwin')
    if hasattr(module_list, 'SwinTransformer'):
        delattr(module_list, 'SwinTransformer')

# ============================ 
# SWIN TRANSFORMER COMPONENTS
# ============================ 
class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        attn = self.softmax(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x

class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4.0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim)
        )

    def forward(self, x):
        H, W = self.H, self.W
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = F.pad(x, (0, 0, pad_l, pad_r, pad_t, pad_b))
        _, Hp, Wp, _ = x.shape
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x
        x_windows = self.window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = self.window_reverse(attn_windows, self.window_size, Hp, Wp)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W, :].contiguous()
        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

    def window_partition(self, x, window_size):
        B, H, W, C = x.shape
        x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
        windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
        return windows

    def window_reverse(self, windows, window_size, H, W):
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
        return x

class _SwinTransformerCore(nn.Module):
    def __init__(self, c1, c2, num_heads=4, depth=2, window_size=7):
        super().__init__()
        self.c2 = c2
        self.input_proj = nn.Conv2d(c1, c2, kernel_size=1)
        self.blocks = nn.ModuleList()
        for i in range(depth):
            shift_size = 0 if (i % 2 == 0) else window_size // 2
            self.blocks.append(
                SwinTransformerBlock(dim=c2, num_heads=num_heads, 
                                   window_size=window_size, shift_size=shift_size)
            )
        self.norm = nn.LayerNorm(c2)
        self.output_proj = nn.Conv2d(c2, c2, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.input_proj(x)
        x = x.flatten(2).transpose(1, 2)
        for block in self.blocks:
            block.H, block.W = H, W
            x = block(x)
        x = self.norm(x)
        x = x.transpose(1, 2).reshape(B, self.c2, H, W)
        x = self.output_proj(x)
        return x

class UAVSwin(nn.Module):
    # Ultralytics thường truyền (in_channels, out_channels, *args_từ_yaml)
    def __init__(self, c1, c2, *args):
        super().__init__()
        
        # --- DEBUG ĐỂ KIỂM TRA LỖI KÊNH (CHANNELS) ---
        print(f"🐞 DEBUG INIT UAVSwin: c1={c1}, c2={c2}, args={args}")
        
        # Xử lý các tham số được truyền vào từ YAML
        if len(args) == 3:
            self.num_heads, self.depth, self.window_size = args
        elif len(args) == 2:
            # Xử lý trường hợp Ultralytics bị nhầm c2 thành args[0]
            self.num_heads = c2
            self.depth, self.window_size = args
            c2 = c1 
        else:
            self.num_heads = args[0] if len(args) > 0 else 4
            self.depth = args[1] if len(args) > 1 else 2
            self.window_size = args[2] if len(args) > 2 else 7

        self.c2 = c2 # Gán output channels
        
        print(f"🔧 UAVSwin setup: Input channels={c1} -> Output channels={self.c2} | Heads={self.num_heads}, Depth={self.depth}")
        
        self.core = _SwinTransformerCore(
            c1=c1,
            c2=self.c2,
            num_heads=self.num_heads,
            depth=self.depth,
            window_size=self.window_size
        )

    def forward(self, x):
        return self.core(x)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys
from ultralytics import YOLO
import ultralytics.nn.modules as modules
import ultralytics.nn.modules.block as block
import ultralytics.nn.tasks as tasks
import torch.nn as nn

# ============================
# 2. ĐĂNG KÝ VÀ CHẠY
# ============================
# ✅ ĐĂNG KÝ VỚI TÊN MỚI
modules.UAVSwin = UAVSwin
block.UAVSwin = UAVSwin
tasks.UAVSwin = UAVSwin

print("✅ UAVSwin đã được đăng ký!")

# 3. CẤU HÌNH & LOAD MODEL
# ============================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Sử dụng device: {device}")

project_name = 'runs_yolo11s_new'
exp_name = 'swin_load11_new'
save_dir = os.path.join(project_name, exp_name)

# Đường dẫn file của bạn
yaml_path = '/home/nguyen/Downloads/UAV_Slicing/final_dataset/CaiTien.yaml'
data_path = '/home/nguyen/Downloads/UAV_Slicing/dataset.yaml'

# ============================
# LOAD MODEL
# ============================
# Nếu đã có checkpoint -> resume
last_checkpoint = os.path.join(save_dir, "weights", "last.pt")

if os.path.exists(last_checkpoint):
    print(f"🔄 Resume training from checkpoint: {last_checkpoint}")
    model = YOLO(last_checkpoint)     # load đúng last.pt
    resume_flag = True
else:
    print("🚀 Bắt đầu huấn luyện mới...")
    model = YOLO(yaml_path)
    # Load pretrained weights nếu cần
    model.load('yolo11s.pt')  # Uncomment nếu muốn load pretrained
    resume_flag = False

print(f"📄 Model config: {yaml_path}")
print(f"📄 Dataset YAML: {data_path}\n")


if __name__ == '__main__':
    # Load model
    torch.cuda.empty_cache()
    
    # Train
    model.train(
        data=data_path,
        epochs=150,
        optimizer='SGD',
        momentum=0.9,
        lr0=0.001,
        imgsz=640,
        batch=4,
        device=device,
        project=project_name,
        name=exp_name,
        verbose=True,
        amp=False,          # ⚠️ Tắt AMP do GPU T1000 báo lỗi tương thích
        exist_ok=True,
        cache=False,
        resume=resume_flag
    )

✅ UAVSwin đã được đăng ký!
🖥️ Sử dụng device: cuda
🚀 Bắt đầu huấn luyện mới...
WARNING ⚠️ no model scale passed. Assuming scale='s'.
🐞 DEBUG INIT UAVSwin: c1=128, c2=4, args=(2, 7)
🔧 UAVSwin setup: Input channels=128 -> Output channels=128 | Heads=4, Depth=2
Transferred 257/587 items from pretrained weights
📄 Model config: /home/nguyen/Downloads/UAV_Slicing/final_dataset/CaiTien.yaml
📄 Dataset YAML: /home/nguyen/Downloads/UAV_Slicing/dataset.yaml

New https://pypi.org/project/ultralytics/8.4.18 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.6 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA T1000 8GB, 7967MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/nguyen/Downloads/UAV_Slicing/dataset.yaml, degrees=0.0, deterministic

KeyboardInterrupt: 

## Train with Concat

In [ ]:
import os
import torch
from ultralytics import YOLO

if __name__ == '__main__':
    # Định nghĩa device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # --- CẤU HÌNH ĐƯỜNG DẪN ---
    yaml_path = '/home/nguyen/Downloads/UAV_Slicing/final_dataset/caiTien_SwinTrans.yaml'
    data_path = '/home/nguyen/Downloads/UAV_Slicing/dataset.yaml'
    
    # Cấu hình thư mục lưu kết quả
    project_dir = 'runs_yolo11s_custom'
    run_name = 'fusion_uav_detection'
    
    # Đường dẫn trỏ tới file checkpoint cuối cùng
    last_ckpt = os.path.join(project_dir, run_name, 'weights', 'last.pt')
    
    # --- LOGIC HUẤN LUYỆN (TỰ ĐỘNG RESUME HOẶC TRAIN MỚI) ---
    if os.path.exists(last_ckpt):
        print(f"🔄 Tìm thấy checkpoint đang train dở tại: {last_ckpt}")
        print("Đang khôi phục trạng thái và tiếp tục huấn luyện (Resume)...")
        
        # Load mô hình từ file last.pt thay vì file yaml
        model = YOLO(last_ckpt)
        
        # Khi resume=True, YOLO sẽ tự động đọc lại file data, số epoch còn lại, optimizer...
        model.train(resume=True)
        
    else:
        print("🚀 Không tìm thấy file checkpoint. Bắt đầu huấn luyện mới từ đầu...")
        
        # 1. Khởi tạo mô hình từ cấu trúc file YAML
        model = YOLO(yaml_path)
        
        # 2. Load pretrained weights để tăng tốc hội tụ
        model.load('yolo11s.pt') 
        
        # 3. Tiến hành huấn luyện mới
        model.train(
            data=data_path,
            epochs=200,
            optimizer='SGD',
            momentum=0.9,
            lr0=0.001,
            imgsz=640,
            batch=8,
            device=device,
            project=project_dir,
            name=run_name,
            verbose=True,
            amp=False,          # ⚠️ Tắt AMP do GPU T1000 báo lỗi tương thích
            exist_ok=True,
            cache=False,
            resume=False
        )

🚀 Không tìm thấy file checkpoint. Bắt đầu huấn luyện mới từ đầu...
WARNING ⚠️ no model scale passed. Assuming scale='s'.
Transferred 378/563 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.19 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.6 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA T1000 8GB, 7967MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/nguyen/Downloads/UAV_Slicing/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, k

In [ ]:
from ultralytics import YOLO

model = YOLO('/home/nguyen/Downloads/UAV_Slicing/runs_yolo11s_custom/fusion_uav_detection/weights/best.pt')

metrics = model.val(
    data='/home/nguyen/Downloads/UAV_Slicing/dataset.yaml',
    split='test',
    imgsz=640,
    batch=8,
    device='cuda',
    project='./runs_yolo11s_custom',           # lưu cùng thư mục train
    name='yolov11_test',          # thư mục kết quả test
    verbose=True
)

print(metrics)
